<a href="https://colab.research.google.com/github/gunvarawork-cpu/code-repository/blob/main/LAB_3/LAB_Comparison_6606614623.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install earthengine-api geemap -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.1 MB/s eta 0:00:00


In [2]:
import ee
import geemap
import numpy as np

ee.Authenticate()
ee.Initialize(project='ee-gunvarawoon')

In [3]:
# =============================
# ROI
# =============================
roi = ee.Geometry.Polygon([
    [[100.40,14.40],
     [100.80,14.40],
     [100.80,14.10],
     [100.40,14.10],
     [100.40,14.40]]
])


# =============================
# LOAD TRAINING
# =============================
training = ee.FeatureCollection(
    'projects/ee-gunvarawoon/assets/training_data_ayutthaya'
)


# =============================
# IMAGE + MASK
# =============================
def maskS2(image):
    qa = image.select('QA60')
    return image.updateMask(
        qa.bitwiseAnd(1 << 10).eq(0).And(
        qa.bitwiseAnd(1 << 11).eq(0))
    )

image = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(roi)
         .filterDate('2024-01-01','2024-03-31')
         .map(maskS2)
         .median()
         .clip(roi))


# =============================
# FEATURES
# =============================
bands = image.select(['B2','B3','B4','B8','B11'])

ndvi = image.normalizedDifference(['B8','B4']).rename('NDVI')
ndwi = image.normalizedDifference(['B3','B8']).rename('NDWI')
ndbi = image.normalizedDifference(['B11','B8']).rename('NDBI')

features = bands.addBands([ndvi, ndwi, ndbi])


# =============================
# SAMPLE
# =============================
samples = features.sampleRegions(
    collection=training,
    properties=['Class'],
    scale=10
)

samples = samples.randomColumn('random')
train = samples.filter(ee.Filter.lt('random',0.8))
test = samples.filter(ee.Filter.gte('random',0.8))


# =============================
# FUNCTION: evaluate model
# =============================
def evaluate(model, name):
    classified = test.classify(model)
    cm = classified.errorMatrix('Class','classification')

    acc = cm.accuracy().getInfo()
    kappa = cm.kappa().getInfo()

    producers = [p[0] for p in cm.producersAccuracy().getInfo()]
    users = cm.consumersAccuracy().getInfo()[0]

    f1 = []
    for p,u in zip(producers, users):
        if (p+u)==0:
            f1.append(0)
        else:
            f1.append(2*(p*u)/(p+u))

    print(f"\n===== {name} =====")
    print("Accuracy:", acc)
    print("Kappa:", kappa)
    print("F1:", f1)

    return acc, kappa, f1


# =============================
# MODEL 1: RF (50 trees)
# =============================
rf50 = ee.Classifier.smileRandomForest(50).train(
    features=train,
    classProperty='Class',
    inputProperties=features.bandNames()
)

rf50_acc, rf50_kappa, rf50_f1 = evaluate(rf50, "RF (50 trees)")


# =============================
# MODEL 2: RF (100 trees)
# =============================
rf100 = ee.Classifier.smileRandomForest(100).train(
    features=train,
    classProperty='Class',
    inputProperties=features.bandNames()
)

rf100_acc, rf100_kappa, rf100_f1 = evaluate(rf100, "RF (100 trees)")


# =============================
# MODEL 3: Gradient Tree Boost
# =============================
gtb = ee.Classifier.smileGradientTreeBoost(
    numberOfTrees=100,
    shrinkage=0.1,
    maxNodes=20
).train(
    features=train,
    classProperty='Class',
    inputProperties=features.bandNames()
)

gtb_acc, gtb_kappa, gtb_f1 = evaluate(gtb, "Gradient Tree Boost")


# =============================
# SUMMARY COMPARISON
# =============================
print("\n===== FINAL COMPARISON =====")

print("\nRF (50 trees)")
print("Accuracy:", rf50_acc, "| Kappa:", rf50_kappa)

print("\nRF (100 trees)")
print("Accuracy:", rf100_acc, "| Kappa:", rf100_kappa)

print("\nGTB")
print("Accuracy:", gtb_acc, "| Kappa:", gtb_kappa)


# =============================
# MAP COMPARISON
# =============================
Map = geemap.Map()
Map.centerObject(roi, 11)

rf_map = features.classify(rf100)
gtb_map = features.classify(gtb)

Map.addLayer(rf_map,
             {'min':1,'max':5,'palette':['blue','cyan','yellow','green','red']},
             'RF 100')

Map.addLayer(gtb_map,
             {'min':1,'max':5,'palette':['blue','cyan','yellow','green','red']},
             'GTB')

Map


===== RF (50 trees) =====
Accuracy: 0.7938144329896907
Kappa: 0.7311157311157311
F1: [0, 0.9268292682926829, 1.0, 0.8275862068965517, 0.4, 0.25]

===== RF (100 trees) =====
Accuracy: 0.7835051546391752
Kappa: 0.7181013008580127
F1: [0, 0.9268292682926829, 1.0, 0.8070175438596492, 0.3846153846153846, 0.25]

===== Gradient Tree Boost =====
Accuracy: 0.7938144329896907
Kappa: 0.7327455572392891
F1: [0, 0.8717948717948718, 1.0, 0.8214285714285715, 0.5185185185185186, 0.3333333333333333]

===== FINAL COMPARISON =====

RF (50 trees)
Accuracy: 0.7938144329896907 | Kappa: 0.7311157311157311

RF (100 trees)
Accuracy: 0.7835051546391752 | Kappa: 0.7181013008580127

GTB
Accuracy: 0.7938144329896907 | Kappa: 0.7327455572392891


Map(center=[14.250050034795764, 100.59999999999953], controls=(WidgetControl(options=['position', 'transparent…